Configuração do Ambiente

<!-- Bibliotecas nescessarias 
Stable-Baselines3:
pip install stable-baselines3

Gymnasium 
pip install gymnasium

NumPy:
conda install numpy

Matplotlib:
conda install matplotlib -->


In [ ]:
import gymnasium as gym  # gymnasium
from gymnasium import spaces
import numpy as np


class CustomTrafficEnv(gym.Env): #Classe criada que herda de gym.env para criar o ambiente
    def __init__(self):
        super(CustomTrafficEnv, self).__init__()
        
        # Espaço de observação: Simula estado do tráfego (ex.: fluxo nas faixas)
        self.observation_space = spaces.Box(low=0, high=100, shape=(4,), dtype=np.float32)
        
        # Espaço de ação:  3 ações (0 = manter, 1 = aumentar, 2 = diminuir tempo de semáforo)
        self.action_space = spaces.Discrete(3)
        
        # Variáveis iniciais
        self.state = None
        self.done = False

    def reset(self, seed=None, options=None):
        # Configuração seed 
        super().reset(seed=seed)
        # Define o estado inicial com base em cenários variados, escolha aleátoria de cenários
        scenario = np.random.choice(["leve", "moderado", "pico"])
        if scenario == "leve":
            self.state = np.random.randint(0, 30, size=(4,))
        elif scenario == "moderado":
            self.state = np.random.randint(30, 70, size=(4,))
        else:  # Cenário de pico
            self.state = np.random.randint(70, 100, size=(4,))
        
        self.done = False
        return self.state, {} # Retorna o estado inicial e um dicionário vazio para info

    def step(self, action):
        # Aplica a ação e atualiza o estado
        if action == 0:  # Manter
            self.state += np.random.randint(-5, 5, size=(4,))
        elif action == 1:  # Aumentar
            self.state += np.random.randint(-10, 0, size=(4,))
        elif action == 2:  # Diminuir
            self.state += np.random.randint(0, 10, size=(4,))
        
        self.state = np.clip(self.state, 0, 100)
        
        # Calcula a recompensa
        mean_traffic = np.mean(self.state)
        if mean_traffic > 80:  # Congestionamento severo
            reward = -5
        elif 50 < mean_traffic <= 80:  # Congestionamento moderado
            reward = -1
        else:  # Fluxo livre
            reward = 2
        
       # Define a terminação e o truncamento do episódio
        done = np.random.random() > 0.95  # Simulação de fim aleatório
        truncated = False  # Não estamos tratando truncamento, então sempre False

        return self.state, reward, done, truncated, {}  # Retorna 5 valores
        
    def render(self, mode='human'):
        # Exibe o estado atual do tráfego
        print(f"Estado atual do tráfego: {self.state}")

#criar o ambiente
env = CustomTrafficEnv()


TESTANDO AMBIENTE

In [3]:
# Teste do ambiente com 5 valores
env = CustomTrafficEnv()
obs, _ = env.reset()  # Reset retornando 2 valores
print("Estado inicial:", obs)

# Executando ações no ambiente
for _ in range(5):  # Teste com 5 passos
    action = env.action_space.sample()  # Ação aleatória
    obs, reward, done, truncated, info = env.step(action)  # Esperando 5 valores
    print(f"Ação: {action}, Novo estado: {obs}, Recompensa: {reward}, Finalizado? {done}, Truncado? {truncated}")
    if done:
        obs, _ = env.reset()  # Reset após terminar o episódio

Estado inicial: [51 43 48 55]
Ação: 0, Novo estado: [47 38 51 52], Recompensa: 2, Finalizado? False, Truncado? False
Ação: 2, Novo estado: [53 46 58 61], Recompensa: -1, Finalizado? False, Truncado? False
Ação: 2, Novo estado: [58 49 61 70], Recompensa: -1, Finalizado? False, Truncado? False
Ação: 2, Novo estado: [65 55 69 78], Recompensa: -1, Finalizado? False, Truncado? False
Ação: 1, Novo estado: [64 50 64 70], Recompensa: -1, Finalizado? False, Truncado? False


TREINAMENTO PPO

In [5]:
from stable_baselines3 import PPO  #importa o algoritmo
# Configuração e treinamento do modelo PPO
model = PPO("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=10000)

# Salvando o modelo
model.save("ppo_traffic_control")


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 22.8     |
|    ep_rew_mean     | 2.71     |
| time/              |          |
|    fps             | 1043     |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 17.8        |
|    ep_rew_mean          | 13.6        |
| time/                   |             |
|    fps                  | 711         |
|    iterations           | 2           |
|    time_elapsed         | 5           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.013597487 |
|    clip_fraction        | 0.195       |
|    clip_range           | 0.2         |
|    entropy_loss   

AVALIAÇÃO DO MODELO NO AMBIENTE

In [7]:
# Carregar o modelo salvo
model = PPO.load("ppo_traffic_control")

# Avaliação do modelo treinado
obs, _ = env.reset()  # Pegue apenas o estado inicial, ignorando o 'info'
for _ in range(20):  # Execute 20 passos
    action, _ = model.predict(obs)  # Passe apenas o estado (obs) para o modelo
    obs, reward, done, truncated, info = env.step(action)  # Agora recebemos 5 valores
    env.render()
    print(f"Ação: {action}, Recompensa: {reward}")
    if done:
        obs, _ = env.reset()  # Reinicie o ambiente ao final do episódio


Estado atual do tráfego: [23 17  7  1]
Ação: 1, Recompensa: 2
Estado atual do tráfego: [24 18  7  7]
Ação: 2, Recompensa: 2
Estado atual do tráfego: [23 17  4  0]
Ação: 1, Recompensa: 2
Estado atual do tráfego: [14 11  0  0]
Ação: 1, Recompensa: 2
Estado atual do tráfego: [5 9 0 0]
Ação: 1, Recompensa: 2
Estado atual do tráfego: [0 3 0 0]
Ação: 1, Recompensa: 2
Estado atual do tráfego: [0 0 0 0]
Ação: 1, Recompensa: 2
Estado atual do tráfego: [2 4 0 8]
Ação: 2, Recompensa: 2
Estado atual do tráfego: [ 6  0  4 12]
Ação: 0, Recompensa: 2
Estado atual do tráfego: [2 0 0 6]
Ação: 1, Recompensa: 2
Estado atual do tráfego: [0 4 0 9]
Ação: 0, Recompensa: 2
Estado atual do tráfego: [3 1 0 7]
Ação: 0, Recompensa: 2
Estado atual do tráfego: [0 0 0 0]
Ação: 1, Recompensa: 2
Estado atual do tráfego: [0 0 0 0]
Ação: 1, Recompensa: 2
Estado atual do tráfego: [6 0 9 2]
Ação: 2, Recompensa: 2
Estado atual do tráfego: [2 0 3 0]
Ação: 1, Recompensa: 2
Estado atual do tráfego: [0 0 0 0]
Ação: 0, Recompen